# prepare_pods.ipynb

**Run this when you need to materialize per-VM pod folders on `/workspace`.** It
installs git, clones/pulls the repo, and generates the per-VM bundles
`Pod_1..Pod_7` at the `/workspace` root from the single `Pod/` template
(setting each one's `VM_NAME`). Because `/workspace` is shared across VMs, once
this finishes every VM already has its folder: VM _N_ just opens `/workspace/Pod_N`.

By default this notebook **does not overwrite existing `Pod_N` folders**. That is
intentional: if `Pod_1..Pod_5` are already in use, re-running this can safely add
missing `Pod_6` / `Pod_7` without disturbing them. Set `OVERWRITE_EXISTING=True`
only when you intentionally want to rebuild all generated bundles.


In [ ]:
# ---- config + git install + clone/pull (setup logic) ----
import os, subprocess

URL = 'https://github.com/Nice9Tian/stable-query-latent.git'
REPO = '/workspace/stable-query-latent'
DEST_ROOT = '/workspace'      # shared FS -> all VMs see the generated Pod_N folders
POD_NUMBERS = range(1, 8)     # Pod_1 -> VM1, ..., Pod_7 -> VM7
VM_PREFIX = 'VM'
OVERWRITE_EXISTING = False    # keep existing Pod_1..5 safe by default

def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)

sh('type -p git >/dev/null 2>&1 || (apt-get update && apt-get install -y git)')
if not os.path.isdir(os.path.join(REPO, '.git')):
    sh(f'git clone {URL} {REPO}')
sh(f'cd {REPO} && git remote set-url origin {URL} && git pull origin main')
sh(f'cd {REPO} && git rev-parse --short HEAD')


In [ ]:
# ---- build missing Pod_N bundles at /workspace from the Pod/ template ----
import json, re, shutil
from pathlib import Path

TEMPLATE = Path(REPO) / 'Pod'
NBS_WITH_VMNAME = ['training.ipynb', 'prepare_training.ipynb', 'realtime_reader.ipynb']
VM_RE = re.compile(r'VM_NAME = "[^"]*"')
COPY_IGNORE = shutil.ignore_patterns('__pycache__', 'Pod_[0-9]*')

def set_vm_name(nb_path, vm):
    doc = json.loads(Path(nb_path).read_text(encoding='utf-8'))
    hits = 0
    for cell in doc.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        src = cell['source']
        joined = ''.join(src) if isinstance(src, list) else src
        if VM_RE.search(joined):
            cell['source'] = VM_RE.sub(f'VM_NAME = "{vm}"', joined, count=1)
            hits += 1
    Path(nb_path).write_text(json.dumps(doc, ensure_ascii=False, indent=1) + '\n', encoding='utf-8')
    return hits

assert TEMPLATE.is_dir(), f'template not found: {TEMPLATE} (did the pull succeed?)'
built = []
skipped = []
for n in POD_NUMBERS:
    vm = f'{VM_PREFIX}{n}'
    dest = Path(DEST_ROOT) / f'Pod_{n}'
    if dest.exists():
        if not OVERWRITE_EXISTING:
            skipped.append((str(dest), vm))
            print(f'skip existing {dest}  (VM_NAME={vm})', flush=True)
            continue
        shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(TEMPLATE, dest, ignore=COPY_IGNORE)
    for nb in NBS_WITH_VMNAME:
        h = set_vm_name(dest / nb, vm)
        assert h == 1, f'{dest / nb}: expected 1 VM_NAME line, found {h}'
    built.append((str(dest), vm))
    print(f'built {dest}  (VM_NAME={vm})', flush=True)

print('\nDone. On each VM open its folder, then run:')
print('   setup.ipynb  ->  prepare_training.ipynb  ->  training.ipynb  ->  check_paralle.ipynb')
if built:
    print('\nBuilt:')
    for d, vm in built:
        print(f'   {vm}: {d}')
if skipped:
    print('\nSkipped existing (set OVERWRITE_EXISTING=True to rebuild intentionally):')
    for d, vm in skipped:
        print(f'   {vm}: {d}')
